# Prueba técnica – Científico/a de Datos INE
**Nilton Reis**


## Conexión a la base de datos

In [11]:
%pip install mysql-connector-python requests

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="sql.freedb.tech", port=3306,
    user="u_y8tlWZ", password="wSkxw3xbhsQg",
    database="freedb_N4nnid0a",
)

# 1. Clasificando texto

### 1.1 Descargar train

In [13]:
train = pd.read_sql("SELECT * FROM train", conn)
print(train.shape)
train.head()

C:\Users\Annia\AppData\Local\Temp\ipykernel_7480\1121972671.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  train = pd.read_sql("SELECT * FROM train", conn)


(35340, 5)


,id,actividad,oficio,tareas,cod_final
0,0,producción uva exportación,temporera agrícola,mantenimiento de parronales,01
1,1,corporación municipal,profesor mecánico industrial,"preparar clases, enseñar",84
2,4,contratista de rrhh para obras de construcción,trabajador de construcción,veredas,41
3,5,municipalidad de puerto natales,técnico paramedico,"toma de signos vitales, entrega de horas a pac...",84
4,7,guardias seguridad y proteccion,administrador de contrato,administrar dirigir contratar personal,80


In [14]:
# Exploración rápida: tipos, nulos y distribución de clases
train.info()
print(train.isna().sum())
print("N° de clases:", train["cod_final"].nunique())
train["cod_final"].value_counts().head(15)

<class 'pandas.DataFrame'>
RangeIndex: 35340 entries, 0 to 35339
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         35340 non-null  int64
 1   actividad  35340 non-null  str  
 2   oficio     35340 non-null  str  
 3   tareas     35340 non-null  str  
 4   cod_final  35340 non-null  str  
dtypes: int64(1), str(4)
memory usage: 1.3 MB
id           0
actividad    0
oficio       0
tareas       0
cod_final    0
dtype: int64
N° de clases: 79


cod_final
48    5507
01    3629
84    3495
85    3189
86    1682
49    1598
41    1546
10    1327
56    1137
81     799
43     796
04     726
45     616
96     495
03     435
Name: count, dtype: int64

**Comentario (exploración):** _escribe aquí: cuántas clases hay, si están desbalanceadas, si hay nulos en el texto._

### 1.2 Preparación del texto y entrenamiento

In [15]:
import unicodedata, re

def limpiar(t):
    t = str(t).lower()
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()  # quita tildes
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def preparar(df):
    cols = ["actividad", "oficio", "tareas"]
    texto = df[cols].fillna("").astype(str).agg(" ".join, axis=1)
    return texto.apply(limpiar)

X_train = preparar(train)
y_train = train["cod_final"].astype(str).str.zfill(2)   # el código es una categoría, no un número ("01")
X_train.head()

0    produccion uva exportacion temporera agricola ...
1    corporacion municipal profesor mecanico indust...
2    contratista de rrhh para obras de construccion...
3    municipalidad de puerto natales tecnico parame...
4    guardias seguridad y proteccion administrador ...
dtype: str

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier

modelo = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LinearSVC(class_weight="balanced")),
])
modelo.fit(X_train, y_train)

base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

**Comentario (decisiones):** _escribe con tus palabras por qué: juntar actividad + oficio + tareas, limpiar tildes, TF-IDF con bigramas, LinearSVC (rápido y bueno para texto con muchas clases), class_weight balanced (clases desbalanceadas), y por qué no optimicé (tiempo)._

### 1.3 Descargar test y predecir

In [19]:
def leer_tabla(nombre):
    c = mysql.connector.connect(
        host="sql.freedb.tech", port=3306,
        user="u_y8tlWZ", password="wSkxw3xbhsQg",
        database="freedb_N4nnid0a",
    )
    df = pd.read_sql(f"SELECT * FROM {nombre}", c)
    c.close()
    return df

In [23]:
import time

def leer_tabla(nombre, intentos=6, espera=10):
    """Lee una tabla con una conexión nueva. Reintenta porque el servidor gratuito
    limita las conexiones simultáneas (lo usan varios postulantes a la vez)."""
    for i in range(intentos):
        try:
            c = mysql.connector.connect(
                host="sql.freedb.tech", port=3306,
                user="u_y8tlWZ", password="wSkxw3xbhsQg",
                database="freedb_N4nnid0a",
            )
            df = pd.read_sql(f"SELECT * FROM {nombre}", c)
            c.close()
            return df
        except mysql.connector.Error as e:
            print(f"Intento {i+1} falló: {e}. Reintento en {espera} s...")
            time.sleep(espera)
    raise RuntimeError("No se pudo conectar a la base")

In [25]:
test = leer_tabla("test")
X_test = preparar(test)
y_test = test["cod_final"].astype(str).str.zfill(2)
y_pred = modelo.predict(X_test)
print(test.shape)

Intento 1 falló: 1203 (42000): User u_y8tlWZ already has more than 'max_user_connections' active connections. Reintento en 10 s...
Intento 2 falló: 1203 (42000): User u_y8tlWZ already has more than 'max_user_connections' active connections. Reintento en 10 s...


C:\Users\Annia\AppData\Local\Temp\ipykernel_7480\399275523.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {nombre}", c)


(3927, 5)


### 1.4 Resultados

In [26]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Base   -> accuracy:", round(accuracy_score(y_test, base.predict(X_test)), 3),
      "| macro F1:", round(f1_score(y_test, base.predict(X_test), average="macro"), 3))
print("Modelo -> accuracy:", round(accuracy_score(y_test, y_pred), 3),
      "| macro F1:", round(f1_score(y_test, y_pred, average="macro"), 3))
print(classification_report(y_test, y_pred, zero_division=0))

Base   -> accuracy: 0.156 | macro F1: 0.004
Modelo -> accuracy: 0.89 | macro F1: 0.751
              precision    recall  f1-score   support

          01       0.96      0.95      0.96       403
          02       0.79      0.97      0.87        35
          03       0.85      0.92      0.88        48
          04       0.96      0.94      0.95        81
          06       0.67      1.00      0.80         2
          07       0.75      0.60      0.67         5
          08       0.75      0.90      0.82        10
          09       0.50      0.30      0.38        10
          10       0.88      0.90      0.89       147
          11       0.65      0.73      0.69        15
          12       0.50      1.00      0.67         1
          14       0.79      0.89      0.84        38
          16       0.74      0.90      0.81        29
          17       0.89      0.94      0.92        18
          18       0.67      0.75      0.71         8
          19       0.00      0.00      0.00     

In [27]:
# Clases donde le va peor (F1 por clase, con su soporte)
rep = pd.DataFrame(classification_report(y_test, y_pred, zero_division=0, output_dict=True)).T
rep = rep.drop(index=["accuracy", "micro avg", "macro avg", "weighted avg"], errors="ignore")
rep.sort_values("f1-score").head(10)

,precision,recall,f1-score,support
19,0.000000,0.000000,0.000000,1.0
27,0.000000,0.000000,0.000000,5.0
24,0.000000,0.000000,0.000000,1.0
59,0.000000,0.000000,0.000000,2.0
70,0.000000,0.000000,0.000000,1.0
09,0.500000,0.300000,0.375000,10.0
32,0.444444,0.400000,0.421053,10.0
33,0.555556,0.555556,0.555556,27.0
20,0.666667,0.500000,0.571429,8.0
71,0.566667,0.586207,0.576271,29.0


In [28]:
# Confusiones más frecuentes y ejemplos de errores
err = test.assign(real=y_test, pred=y_pred).query("real != pred")
print(err.groupby(["real", "pred"]).size().sort_values(ascending=False).head(10))
err[["actividad", "oficio", "tareas", "real", "pred"]].head(10)

real  pred
43    41      14
48    01      12
84    86      10
      85      10
41    43       9
86    84       5
48    10       5
85    84       5
84    71       4
48    16       4
dtype: int64


,actividad,oficio,tareas,real,pred
1,metro de santiago,mantencion en metro,realiza mantencion a tableros eléctricos y bom...,49,43
18,producción y procesamiento de salmón,subgerente de planta,"administra, realiza el funcionamiento del. are...",03,10
31,"presta sevicion en desmurres, confeccion y rep...",labores agricolas,"desmurres, confección y reparación de cercos.",01,14
33,productora de eventos,garzona,"montar mesas, servir platos, limpiar loza",93,56
35,confeccion de artesania con conchas de mar,artesana en conchitas de mar,"buscar conchitas de mar, hacer llaveros",32,14
36,comerciante,comerciante,carro de mote/huesillo en la feria,48,56
51,recoleccion y venta de chatarra y aluminio,venta de chatarra y latas de aluminio,recolección de latas y chatarra,48,38
76,servicio público,chofer de bus,conduce transporte estatal,84,49
79,traslado de personal,auxiliar de aseo,limpiar y ordenar baños y galpones,49,78
81,repaciones menor en viviendas particular,maestro,"reparaciones menor en viviendas, reparación de...",41,43


### 1.5 Análisis
El modelo logra accuracy de 0,89 y macro f1 de 0,75, contra 0,16 y de 0,004 del modelo base. le va muy bien en clases grandes como la 01, pero falla en clases chicas, algunas con 1 o 2 casos de test.
Ademas al revisar los errores note que el codigo sigue las actividades del empleador, mientras que el modelo se guia por el oficio, como por ejemplo un mantenedor electrico del metro tiene un codigo 49 de trasporte, mientras que el modelo lo cásifico como 43de construcción.

### 1.6 Plan de mejora (sin ejecutar)
1. vectorizas las actividades por separado del oficio y las tareas para que el modelo aprenda que las actividades pesan mas que el codigo, admeas por los errores que revise, es lo que mas mejoraria el resultado.
2. agregar n.grasmas de caracter para tolerar errores de tipo "repaciones" en vez de repacariones, que el modelo actual no reoconoce.
3. ajustar los hiperparámetros dcon validación cruzada sobre el train, sin tocar el test.
4. revisar con los codificadores los casos ambiguos, como municipalidades y salud, y conseguir mas ejemplos de las clases chicas.
5. probar el umbral de confianza, los casos dudosos van a codificacion manual en vez de asignarlos automaticamente.

# 2. Modelo en produccion

para los resumenes use openrouter con el modelo que pedian (gemini 2-5 flash lite) y deje el razonamiento apagadopara que salieran mas barato. no escribí la apy key en el código, el notebook la pide al correrla, asi no quedan en github. al principio me daba eror 401 y era porque la llave habia cambiado. antes de procesar las 10 noticias probe con una sola, al final revise que ningun reumen pasara las 50 palabras. todos quedaron en 31 y en 46


# 3. Procesando texto (resúmenes con LLM)

In [30]:
noticias = leer_tabla("noticias")
print(noticias.shape)
noticias.head(2)

C:\Users\Annia\AppData\Local\Temp\ipykernel_7480\399275523.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {nombre}", c)


(10, 2)


,cuerpo,titular
0,fue rodríguez zapatero el que empezó a utiliza...,escenas de geometría variable
1,la reunión que mantendrán este viernes el dipu...,iglesias se reunirá con junqueras sin cristal ...


In [35]:
import requests, getpass

# 1) Cargar la API key (se pide al ejecutar, no queda en el código)
API_KEY = getpass.getpass("OpenRouter API key: ").strip()
print("Largo de la key:", len(API_KEY), "| Empieza con:", API_KEY[:6])

# 2) Función que resume una noticia con el LLM
def resumir(texto):
    prompt = ("Resume la siguiente noticia en español, en un máximo de 50 palabras. "
              "Responde solo con el resumen, sin títulos ni comentarios.\n\n"
              f"Noticia:\n{texto}")
    r = requests.post(
        "https://openrouter.ai/api/v1/responses",
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json={
            "model": "google/gemini-2.5-flash-lite",
            "input": prompt,
            "reasoning": {"effort": "none"},   # razonamiento desactivado
            "max_output_tokens": 200,
        },
        timeout=60,
    )
    if r.status_code != 200:
        print("Error", r.status_code, ":", r.text[:300])   # muestra el motivo exacto
        r.raise_for_status()
    return r.json()["output"][0]["content"][0]["text"].strip()

# 3) Probar con UNA noticia
print("Prueba:", resumir(noticias.loc[0, "cuerpo"]))

# 4) Si la prueba funcionó, resumir las 10 y validar el largo
noticias["resumen"] = noticias["cuerpo"].apply(resumir)
noticias["n_palabras"] = noticias["resumen"].str.split().str.len()
print("¿Todos ≤ 50 palabras?:", (noticias["n_palabras"] <= 50).all())
noticias[["titular", "resumen", "n_palabras"]]

Largo de la key: 73 | Empieza con: sk-or-
Prueba: Sánchez emula la "geometría variable" de Zapatero. Busca apoyos diversos para gobernar, con nacionalistas y ciudadanos como socios clave. Aunque evitan vetos, la inestabilidad pasada y la futura crisis económica son riesgos.
¿Todos ≤ 50 palabras?: True


,titular,resumen,n_palabras
0,escenas de geometría variable,"Sánchez replica la ""geometría variable"" de Zap...",45
1,iglesias se reunirá con junqueras sin cristal ...,Iglesias se reunirá con Junqueras en prisión s...,41
2,bankia paga otros 219 millones al estado vía d...,Bankia pagará 219 millones de euros al Estado ...,39
3,sánchez tiene ya una mayoría en el congreso pa...,La nueva legislatura con mayoría de izquierdas...,40
4,un presidente irresponsable,El rey emérito Juan Carlos I regulariza casi 4...,40
5,castro embarga el palacete y otra finca a mata...,"El juez del caso Nóos, José Castro, ordenó el ...",46
6,"el jefe del ejecutivo, dispuesto a reunirse co...",Rajoy está dispuesto a reunirse con Pablo Igle...,31
7,el último choque con puigdemont y las diferenc...,El juicio del Procés comenzará en febrero en e...,39
8,interior jubila al policía que perdió la visió...,"Un agente antidisturbios de 42 años, que sufri...",39
9,"otegi sobre bateragune ""ni nos domaron, ni nos...",El Tribunal Supremo reabrirá el juicio contra ...,42


In [32]:
import os, requests, getpass

# La API key se pide al ejecutar: NO queda escrita en el código ni se sube a GitHub
API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass.getpass("OpenRouter API key: ")

def resumir(texto):
    prompt = ("Resume la siguiente noticia en español, en un máximo de 50 palabras. "
              "Responde solo con el resumen, sin títulos ni comentarios.\n\n"
              f"Noticia:\n{texto}")
    r = requests.post(
        "https://openrouter.ai/api/v1/responses",
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json={
            "model": "google/gemini-2.5-flash-lite",
            "input": prompt,
            "reasoning": {"effort": "none"},   # razonamiento desactivado
            "max_output_tokens": 200,
        },
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["output"][0]["content"][0]["text"].strip()

# Probar con una noticia antes de correr las 10
API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass.getpass("OpenRouter API key: ").strip()

In [36]:
noticias["resumen"] = noticias["cuerpo"].apply(resumir)
noticias["n_palabras"] = noticias["resumen"].str.split().str.len()
print("¿Todos ≤ 50 palabras?:", (noticias["n_palabras"] <= 50).all())
noticias[["titular", "resumen", "n_palabras"]]

¿Todos ≤ 50 palabras?: True


,titular,resumen,n_palabras
0,escenas de geometría variable,"Sánchez busca reeditar la ""geometría variable""...",41
1,iglesias se reunirá con junqueras sin cristal ...,Iglesias y Junqueras se reunirán sin restricci...,27
2,bankia paga otros 219 millones al estado vía d...,Bankia pagará 219 millones de euros al Estado ...,33
3,sánchez tiene ya una mayoría en el congreso pa...,La mayoría de izquierdas en el Congreso facili...,40
4,un presidente irresponsable,"Juan Carlos I regulariza casi 4,5 millones de ...",41
5,castro embarga el palacete y otra finca a mata...,El juez José Castro ordena el embargo de biene...,50
6,"el jefe del ejecutivo, dispuesto a reunirse co...",Rajoy se ofrece a reunirse con Pablo Iglesias ...,46
7,el último choque con puigdemont y las diferenc...,El juicio del procés en el Tribunal Supremo co...,42
8,interior jubila al policía que perdió la visió...,"Un policía de 42 años, agente de la UIP, se ju...",45
9,"otegi sobre bateragune ""ni nos domaron, ni nos...",El Tribunal Supremo ordenó un nuevo juicio con...,45


**Comentario:** _escribe por qué validas el largo, por qué la clave no va en el código, y qué harías si un resumen supera 50 palabras (reintentar o recortar)._

# 4. Cuéntanos sobre un proyecto

En mi antiguo empleo existian un problema, la gerencia de mantenimiento no tenia como identificar patrones de fallas en los activos de las tiendas (Walmart Chile). al revisar los datos, note que en los ticket de mantenimiento existian en la nube pdf que eran fotograías de documentos de papel y escritos a manos que describian lo que hacían, la principal falla entre otras informaciones. Este problema era relevante ya que para validar el pago del servisdio de proveedores debia el equipo de pago revisar uno a uno cada uno de estos dociumentos. la cantidad de archivos era de 5.000 documentos por mes y cada mes el equipo debia revisar toda esa información. Al identificar el problema realice un multimodelo OCR + IA (Gemini flash lite) la lectura masiva de esos documentos, realizando ua prueba primero con 100 pdf. 
Arme un caso de negocio para este proyecto para ver el o los impactos de negocio para la comparía, estos fueorn 3 impactos:
1. reducir la cantidad de HH dedicadas solo a revisar estos documeentos, bajando en un 54% en tiempo.
2. detecte un cuello de botella por parte del equipo de pago, ya que el 50% de los cierres para pago lo realizaban la ultima semana de cada mes. elaboramos un plan de reducción.
3. Hice una simulacion Monte Carlo con 10.000 escenarios posibles y se proyecto una reduccion del gasto entre 4 al 7% proyectado a 5 años.

Este proyecto lo desarrolle junto con el equipo de gocierno de datos de Walmart Chile usando Vertex AI y leyendo de manera masiva mas de un millon de PDF y subiendo esta informacion a GCP.